In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


In [ ]:


load_dotenv()

model= ChatOpenAI(
    model="google/gemini-2.5-flash-lite",
    max_tokens=1000,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

response = model.invoke("Hello! Are you working?")
print(response.content)


In [ ]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [ ]:
def create_outline(state: BlogState) -> BlogState:
    title= state['title']
    prompt = f'Generate an outline for a blog on the topic - {title}'
    outline=str(model.invoke(prompt).content)
    state['outline'] = outline
    return state

In [ ]:
def create_blog(state: BlogState) -> BlogState:
    title= state['title']
    outline= state['outline']
    prompt = f'write a detailed  blog on the topic - {title} using the folwing outline {outline} '
    content=str(model.invoke(prompt).content)

    state['content'] = content
    return state


In [ ]:
graph = StateGraph(BlogState)

graph.add_node("create_outline",create_outline)
graph.add_node("create_blog",create_blog)



graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)
workflow =graph.compile()

NameError: name 'StateGraph' is not defined

In [ ]:
initial_state= {'title':'Rise of AI in pakistan'}
final_state= workflow.invoke(initial_state)
print(final_state)